# magic-pinecone：在 Kaggle GPU 上跑模型服務並對外提供

這份 notebook 在兩張 T4 上各啟動一個 llama-server：第一張跑微調後的 Gemma 4 E2B v6（4-bit GGUF，聊天用），第二張跑 EmbeddingGemma（知識庫的向量搜尋用）。兩個服務前面放一個小的反向代理，只用一個網址對外：`/v1/embeddings` 轉給 EmbeddingGemma，其餘（`/v1/chat/completions`、`/v1/models`、`/health`）轉給聊天模型。再經通道對外，讓後端的 `/api/v1/assistant/ask` 與知識庫搜尋呼叫。通道有兩種，在設定格用 `TUNNEL` 選：

- `pangolin`：接到自己架的 Pangolin，網址固定。
- `ngrok`：備援。

設定格的 `EMBEDDINGS = False` 可以不啟動 EmbeddingGemma，代理只轉給聊天模型，其他照常運作。

## 開始前（都在 Kaggle 右側面板）

1. **Session options → Accelerator** 選 `GPU T4 x2`，**Internet** 開啟（帳號需完成手機驗證）。
2. **Add-ons → Secrets** 新增 secret，並勾選附加到這份 notebook。
   - 一定要有：
     - `HF_TOKEN`：Hugging Face 的讀取 token（模型 repo 是私有的）。
     - `ASSISTANT_API_KEY`：自己取一串至少 16 字元的隨機字串，後端要用同一串。網址是公開的，沒有這把鑰匙任何人都能用。聊天與向量兩個服務共用這一把。
   - 用 Pangolin 時：
     - `PANGOLIN_ENDPOINT`：Pangolin 的網址，例如 `https://pangolin.example.com`。
     - `PANGOLIN_SITE_ID`、`PANGOLIN_SITE_SECRET`：在 Pangolin 新增 site 時給的那一組。
     - `PANGOLIN_PUBLIC_URL`：指向這個 site 的 resource 網址，例如 `https://llm.example.com`。
   - 用 ngrok 時：`NGROK_AUTHTOKEN`；選用 `NGROK_DOMAIN`（固定網域）。
   - 選用 `HF_REPO`：模型 repo 的「HF帳號/repo名稱」，也可以直接填在設定格。
3. 聊天模型預設是 v6（`gemma4-e2b-v6-Q4_K_M.gguf`，約 3.4 GB）。這個檔案要先從 Google Drive 的 `magic-pinecone/` 資料夾上傳到私有的 HF repo，下載那一格才找得到；找不到時會停下來並提示，也可以把設定格的 `GGUF_FILE` 改回 `e2b-v3-Q4_K_M.gguf`。EmbeddingGemma 用公開的 `ggml-org/embeddinggemma-300M-GGUF`，不用上傳。
4. Pangolin 那邊要先設好（只需一次）：
   - 新增一個 site，記下 ID 與 secret。
   - 新增一個公開的 HTTP resource，target 設為這個 site 上的 `localhost:8080`（http）。這是代理的埠，聊天和向量都走這一個 resource，不用另外設。
   - **關掉這個 resource 的登入驗證**（SSO、密碼、PIN）。後端是程式呼叫，被導去登入頁就會失敗；保護由 `ASSISTANT_API_KEY` 負責。
5. 由上到下執行。第一次要下載 llama.cpp 預編譯檔（約 0.8 GB）、聊天模型（約 3.4 GB）和 EmbeddingGemma（約 0.33 GB），demo 前請提早開。

notebook 不會印出任何 secret，用 Pangolin 時也不會印出網址。

In [ ]:
# 設定
HF_REPO = "conashin/magic-pinecone-gemma4-e2b-gguf"          # 填「HF帳號/repo名稱」；留空則改讀 Kaggle secret HF_REPO
GGUF_FILE = "gemma4-e2b-v6-Q4_K_M.gguf"
CHAT_ALIAS = "e2b"
EMBEDDINGS = True
EMBED_REPO = "ggml-org/embeddinggemma-300M-GGUF"
EMBED_FILE = "embeddinggemma-300M-Q8_0.gguf"
EMBED_ALIAS = "embeddinggemma"
LLAMA_TAG = "b12340"   # 有官方預編譯檔的版本；後端端到端測試用的也是這版
CUDA_BUILD = "12.8"    # 預編譯檔的 CUDA 版本，可選 "12.8" 或 "13.5"
PORT = 8080
CHAT_PORT = 8081
EMBED_PORT = 8082
CTX = 16384            # 兩個同時請求平分，各 8192
PARALLEL = 2
TUNNEL = "pangolin"    # "pangolin" 或 "ngrok"

In [ ]:
# 確認有 GPU
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
# 讀取 secret（只放進變數，不顯示）
from kaggle_secrets import UserSecretsClient

_s = UserSecretsClient()
HF_TOKEN = _s.get_secret("HF_TOKEN")
API_KEY = _s.get_secret("ASSISTANT_API_KEY")

def _opt(name):
    try:
        return _s.get_secret(name)
    except Exception:
        return None

HF_REPO = HF_REPO or _opt("HF_REPO")
assert HF_REPO and "/" in HF_REPO, "請在設定格填 HF_REPO，或新增 Kaggle secret HF_REPO"
assert len(API_KEY) >= 16, "ASSISTANT_API_KEY 太短，請用至少 16 個字元"

assert TUNNEL in ("pangolin", "ngrok")
if TUNNEL == "pangolin":
    PANGOLIN_ENDPOINT = _s.get_secret("PANGOLIN_ENDPOINT")
    PANGOLIN_SITE_ID = _s.get_secret("PANGOLIN_SITE_ID")
    PANGOLIN_SITE_SECRET = _s.get_secret("PANGOLIN_SITE_SECRET")
    PANGOLIN_PUBLIC_URL = _s.get_secret("PANGOLIN_PUBLIC_URL").rstrip("/")
else:
    NGROK_AUTHTOKEN = _s.get_secret("NGROK_AUTHTOKEN")
    NGROK_DOMAIN = _opt("NGROK_DOMAIN")
print("secrets 已讀取；通道：", TUNNEL)

In [ ]:
# 下載 llama.cpp 官方預編譯的 CUDA 版（不用編譯）。已下載就跳過。
import os, subprocess, glob

LLAMA_DIR = "/kaggle/working/llama-bin"
BASE = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
FILES = [f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-{CUDA_BUILD}-x64.tar.gz",        # 程式
         f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-{CUDA_BUILD}-x64.tar.gz"]  # CUDA 執行階段程式庫

def find_server():
    hits = glob.glob(f"{LLAMA_DIR}/**/llama-server", recursive=True)
    return hits[0] if hits else None

if not find_server():
    os.makedirs(LLAMA_DIR, exist_ok=True)
    for f in FILES:
        r = subprocess.run(f"curl -fsSL {BASE}/{f} | tar -xz -C {LLAMA_DIR}",
                           shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-2000:])
            raise RuntimeError("下載或解壓失敗：" + f)

SERVER = find_server()
assert SERVER, "解壓後找不到 llama-server"
libdirs = sorted({os.path.dirname(x) for x in glob.glob(f"{LLAMA_DIR}/**/*.so*", recursive=True)})
os.environ["LD_LIBRARY_PATH"] = ":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")])

r = subprocess.run([SERVER, "--list-devices"], capture_output=True, text=True)
out = (r.stdout + r.stderr)[-1500:]
print(out)
assert r.returncode == 0, "llama-server 無法執行，請把上面的訊息貼回去"
if "CUDA" not in out:
    print("警告：沒有偵測到 CUDA 裝置，模型會改用 CPU，速度會慢很多。請把上面的訊息貼回去。")
print("llama-server 就緒：", SERVER)

In [ ]:
# 下載模型（約 3.4 GB）
from huggingface_hub import hf_hub_download, list_repo_files

if GGUF_FILE not in list_repo_files(HF_REPO, token=HF_TOKEN):
    raise RuntimeError(f"私有 HF repo 裡找不到 {GGUF_FILE}。請先把它從 Google Drive 的 magic-pinecone/ 上傳到私有 repo，"
                       "或把設定格的 GGUF_FILE 改回 \"e2b-v3-Q4_K_M.gguf\"。")
MODEL_PATH = hf_hub_download(HF_REPO, GGUF_FILE, token=HF_TOKEN, local_dir="/kaggle/working/model")
print(MODEL_PATH, round(os.path.getsize(MODEL_PATH) / 1e9, 2), "GB")

In [ ]:
# 下載 EmbeddingGemma（約 0.33 GB）
if EMBEDDINGS:
    EMBED_PATH = hf_hub_download(EMBED_REPO, EMBED_FILE, token=HF_TOKEN, local_dir="/kaggle/working/embed-model")
    print(EMBED_PATH, round(os.path.getsize(EMBED_PATH) / 1e9, 2), "GB")

In [ ]:
# 啟動 llama-server。思考模式一定要關：開著的話提示格式與訓練時不同。
import time, requests

log = open("/kaggle/working/llama-server.log", "w")
env = dict(os.environ, LLAMA_API_KEY=API_KEY, CUDA_VISIBLE_DEVICES="0")   # 用環境變數傳，避免鑰匙出現在行程列表
server = subprocess.Popen(
    [SERVER, "-m", MODEL_PATH, "--host", "127.0.0.1", "--port", str(CHAT_PORT),
     "-ngl", "99", "-c", str(CTX), "-np", str(PARALLEL),
     "--jinja", "--reasoning", "off", "--alias", CHAT_ALIAS],
    stdout=log, stderr=subprocess.STDOUT, env=env,
)

for _ in range(120):
    if server.poll() is not None:
        print(open("/kaggle/working/llama-server.log").read()[-3000:])
        raise RuntimeError("llama-server 啟動失敗")
    try:
        if requests.get(f"http://127.0.0.1:{CHAT_PORT}/health", timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("llama-server 四分鐘內沒有就緒")
print("llama-server 已就緒")

In [ ]:
# 啟動 EmbeddingGemma
embed_server = None
if EMBEDDINGS:
    embed_log = open("/kaggle/working/embed-server.log", "w")
    embed_env = dict(os.environ, LLAMA_API_KEY=API_KEY, CUDA_VISIBLE_DEVICES="1")
    embed_server = subprocess.Popen(
        [SERVER, "-m", EMBED_PATH, "--host", "127.0.0.1", "--port", str(EMBED_PORT),
         "--embeddings", "-c", "2048", "-ub", "2048", "-ngl", "99", "--alias", EMBED_ALIAS],
        stdout=embed_log, stderr=subprocess.STDOUT, env=embed_env,
    )

    for _ in range(120):
        if embed_server.poll() is not None:
            print(open("/kaggle/working/embed-server.log").read()[-3000:])
            raise RuntimeError("embedding llama-server 啟動失敗")
        try:
            if requests.get(f"http://127.0.0.1:{EMBED_PORT}/health", timeout=2).status_code == 200:
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        raise RuntimeError("embedding llama-server 四分鐘內沒有就緒")
    print("embedding llama-server 已就緒")
else:
    print("EMBEDDINGS = False，不啟動 EmbeddingGemma")

In [ ]:
# 啟動反向代理：一個網址，兩個服務
import sys

PROXY_SRC = r'''
import sys
import http.client
import json
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

PORT = int(sys.argv[1])
CHAT_PORT = int(sys.argv[2])
EMBED_PORT = int(sys.argv[3])

HOP = {"connection", "keep-alive", "transfer-encoding", "host", "te", "trailer", "upgrade"}


def is_hop(name):
    n = name.lower()
    return n in HOP or n.startswith("proxy-")


class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        pass

    def forward(self):
        route_embed = EMBED_PORT and self.path.startswith(("/v1/embeddings", "/embeddings"))
        port = EMBED_PORT if route_embed else CHAT_PORT
        length = int(self.headers.get("Content-Length") or 0)
        body = self.rfile.read(length) if length else None
        headers = {k: v for k, v in self.headers.items()
                   if not is_hop(k) and k.lower() != "content-length"}
        conn = http.client.HTTPConnection("127.0.0.1", port, timeout=300)
        try:
            conn.request(self.command, self.path, body=body, headers=headers)
            resp = conn.getresponse()
            data = resp.read()
            status = resp.status
            out = [(k, v) for k, v in resp.getheaders()
                   if not is_hop(k) and k.lower() != "content-length"]
        except (OSError, http.client.HTTPException):
            status = 502
            data = json.dumps({"error": "upstream unreachable"}).encode()
            out = [("Content-Type", "application/json")]
        finally:
            conn.close()
        self.send_response(status)
        for k, v in out:
            self.send_header(k, v)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    do_GET = forward
    do_POST = forward
    do_OPTIONS = forward


ThreadingHTTPServer(("127.0.0.1", PORT), Handler).serve_forever()
'''
open("/kaggle/working/proxy.py", "w").write(PROXY_SRC.lstrip())

proxy_log = open("/kaggle/working/proxy.log", "w")
proxy_proc = subprocess.Popen(
    [sys.executable, "/kaggle/working/proxy.py", str(PORT), str(CHAT_PORT), str(EMBED_PORT if EMBEDDINGS else 0)],
    stdout=proxy_log, stderr=subprocess.STDOUT, start_new_session=True,
)

for _ in range(30):
    if proxy_proc.poll() is not None:
        print(open("/kaggle/working/proxy.log").read()[-3000:])
        raise RuntimeError("代理啟動失敗")
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("代理三十秒內沒有就緒")
print("代理已就緒")

In [ ]:
# 本機測試（經代理）：有鑰匙要成功、沒鑰匙要被拒絕、回答裡不能有思考內容
def ask(base, key, text):
    h = {"Authorization": f"Bearer {key}"} if key else {}
    return requests.post(f"{base}/v1/chat/completions", headers=h, timeout=120, json={
        "model": CHAT_ALIAS, "temperature": 0, "max_tokens": 64,
        "messages": [{"role": "user", "content": text}],
    })

def embed(base, key, text="task: search result | query: 工讀證"):
    h = {"Authorization": f"Bearer {key}"} if key else {}
    return requests.post(f"{base}/v1/embeddings", headers=h, timeout=120,
                         json={"model": EMBED_ALIAS, "input": [text]})

def embed_dim(r):
    try:
        data = r.json()["data"]
        return len(data[0]["embedding"]) if len(data) == 1 else None
    except (ValueError, KeyError, IndexError, TypeError):
        return None

LOCAL = f"http://127.0.0.1:{PORT}"
r = ask(LOCAL, API_KEY, "Reply with the single word: ready")
r.raise_for_status()
msg = r.json()["choices"][0]["message"]
print("回答：", msg["content"])
assert not msg.get("reasoning_content"), "思考模式沒有關掉"
assert ask(LOCAL, None, "hi").status_code == 401, "沒帶鑰匙卻沒被拒絕，請不要開 ngrok"
print("鑰匙檢查通過")
if EMBEDDINGS:
    r = embed(LOCAL, API_KEY)
    r.raise_for_status()
    assert embed_dim(r) == 768, "向量長度不是 768"
    assert embed(LOCAL, None).status_code == 401, "向量服務沒帶鑰匙卻沒被拒絕，請不要開 ngrok"
    print("向量檢查通過：768 維，沒帶鑰匙會被拒絕")

In [ ]:
# 開通道
import shutil, sys

if TUNNEL == "pangolin":
    if not shutil.which("pangolin"):
        # Pangolin 官方安裝腳本
        r = subprocess.run("curl -fsSL https://static.pangolin.net/get-cli.sh | bash",
                           shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-2000:]); print(r.stderr[-2000:])
            raise RuntimeError("Pangolin CLI 安裝失敗")
    PANGOLIN_BIN = shutil.which("pangolin") or next(
        (p for p in ("/usr/local/bin/pangolin", os.path.expanduser("~/.local/bin/pangolin"))
         if os.path.exists(p)), None)
    assert PANGOLIN_BIN, "找不到 pangolin 指令"
    # 用環境變數傳 ID 與 secret；紀錄檔放 /tmp，不進 notebook 輸出
    tunnel_log = open("/tmp/pangolin-site.log", "w")
    tunnel_proc = subprocess.Popen(
        [PANGOLIN_BIN, "up", "site"], stdout=tunnel_log, stderr=subprocess.STDOUT,
        env=dict(os.environ, SITE_ID=PANGOLIN_SITE_ID, SITE_SECRET=PANGOLIN_SITE_SECRET,
                 PANGOLIN_ENDPOINT=PANGOLIN_ENDPOINT))
    PUBLIC_URL = PANGOLIN_PUBLIC_URL
    SHOWN_URL = "<PANGOLIN_PUBLIC_URL>"
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    t = ngrok.connect(PORT, "http", domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(PORT, "http")
    PUBLIC_URL = t.public_url.replace("http://", "https://")
    SHOWN_URL = PUBLIC_URL
    tunnel_proc = None

# 等通道接通，最多兩分鐘
last = "沒有回應"
for _ in range(40):
    if tunnel_proc is not None and tunnel_proc.poll() is not None:
        raise RuntimeError("通道程式結束了，請看 /tmp/pangolin-site.log（可能含網址，不要貼到公開的地方）")
    try:
        r = ask(PUBLIC_URL, API_KEY, "Reply with the single word: ready")
        if r.status_code == 200 and "choices" in r.text:
            if not EMBEDDINGS or embed_dim(embed(PUBLIC_URL, API_KEY)) == 768:
                break
            last = "向量端點沒有回 768 維"
        else:
            last = f"HTTP {r.status_code}"
    except requests.RequestException as e:
        last = type(e).__name__
    time.sleep(3)
else:
    raise RuntimeError(f"經公開網址呼叫失敗（{last}）。用 Pangolin 時請確認 resource 的 target 與登入驗證已關閉。")

print("經公開網址測試成功\n")
print("後端環境變數：")
print(f"ASSISTANT_BASE_URL={SHOWN_URL}/v1")
print(f"ASSISTANT_MODEL={CHAT_ALIAS}")
print("ASSISTANT_API_KEY=<同 Kaggle secret ASSISTANT_API_KEY>")
print("ASSISTANT_DISABLE_THINKING=true")
if EMBEDDINGS:
    print(f"EMBEDDING_BASE_URL={SHOWN_URL}/v1")
    print(f"EMBEDDING_MODEL={EMBED_ALIAS}")
    print("EMBEDDING_API_KEY=<同 Kaggle secret ASSISTANT_API_KEY>")
    print("\n改過 EMBEDDING_* 設定後，請在後端跑一次 reindex（POST /api/v1/knowledge/reindex?wait=true，只能在 compose 網路內呼叫），段落才會有向量。")
    print("沒有設 EMBEDDING_* 時，後端會退回關鍵字搜尋。")

In [ ]:
# 保持執行：這一格要一直跑著，停掉它通道和模型服務仍在，但 Kaggle 可能因閒置回收機器。
# 每分鐘檢查一次，服務掛掉會顯示最後的紀錄。按停止鍵結束。
procs = {"llama-server": (server, "/kaggle/working/llama-server.log"),
         "proxy": (proxy_proc, "/kaggle/working/proxy.log")}
if embed_server is not None:
    procs["embed-server"] = (embed_server, "/kaggle/working/embed-server.log")
try:
    while True:
        dead = [n for n, (p, _) in procs.items() if p.poll() is not None]
        ok = not dead
        try:
            ok = ok and requests.get(f"{LOCAL}/health", timeout=5).status_code == 200
            if embed_server is not None:
                ok = ok and requests.get(f"http://127.0.0.1:{EMBED_PORT}/health", timeout=5).status_code == 200
        except requests.RequestException:
            ok = False
        if tunnel_proc is not None:
            ok = ok and tunnel_proc.poll() is None
        print(time.strftime("%H:%M:%S"), "正常" if ok else "異常", flush=True)
        if not ok:
            for n in (dead or list(procs)):
                print(f"--- {n} ---")
                print(open(procs[n][1]).read()[-2000:])
            break
        time.sleep(60)
except KeyboardInterrupt:
    print("已停止監看")

In [ ]:
# 收尾：關代理、模型服務與通道。用完請在 Kaggle 右上角停止 session，才不會繼續扣 GPU 時數。
proxy_proc.terminate()
if embed_server is not None:
    embed_server.terminate()
server.terminate()
if tunnel_proc is not None:
    tunnel_proc.terminate()
else:
    ngrok.kill()
print("已關閉")